# 18 · One model, a model told the QP, or a specialist?

A frame's slice QP says how hard the codec squeezed it, and the client reads it off every slice header
(desktop-streamer, `stream-h264.js`). A blind model has to hedge between sharpening a clean frame and deblocking a
coded one. Is it worth telling the model the QP, or training a model for high QP alone?

Three models with notebook 7's body (24 channels, 2 residual blocks), trained with the shared recipe
(`training.fit`) on notebook 0's set, which spans slice QP 0 to 51 on both of the host's lines:

1. **Blind**: not told the QP. Notebooks 4 and 8 trained it at three seeds, the ruler for luck.
2. **Told the QP**: each patch's slice QP / 51 as a second input (`models.ToldLuma`), at two seeds.
3. **Specialist**: blind, trained and selected only on patches at slice QP 38 and up.

Each is scored on the validation split, overall and by slice QP bin. A separate high-QP model pays only if the
specialist clearly beats the told model there. A told model ships as the blind graph with its biases a straight line
in the QP (notebook 19), so telling costs the client nothing at run time.

Needs: `00_data_preprocess.ipynb`, and the scores of notebooks 1 (bicubic), 4 and 8 (the blind model at three seeds).

In [1]:
"""Settings."""

import os

import numpy as np
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, training
from helpers.training import DATA_DIR, DEVICE

NOTEBOOK = 18
WIDTH, BLOCKS, STEM = 24, 2, 3   # notebook 7's body
SPECIALIST_QP = (38, 99)         # the slice QP the specialist trains and selects on
TOLD = ("qp",)
COLUMNS = patches.columns(TOLD)
RUN_DIR = training.CHECKPOINT_DIR / "qp_runs"

manifest = training.load_manifest()
SCALE = manifest["scale"]
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]   # notebook 1's scores of the bar
blind = {name: scores for (notebook, name), scores in recorded.items()
         if notebook in (4, 8) and name.startswith("luma 24 + 2 blocks")}   # the blind model at three seeds


def build(told, seed):
    """The body, told or blind; the same seed starts both with the same weights."""
    torch.manual_seed(seed)
    if told:
        return models.ToldLuma(len(COLUMNS), WIDTH, BLOCKS, STEM, scale=SCALE).to(DEVICE)
    return models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE).to(DEVICE)


RUNS = {   # name -> (told, seed, slice QP range, run file)
    "told the QP": (True, 0, None, RUN_DIR / "told_qp_seed0.pt"),
    "told the QP, seed 1": (True, 1, None, RUN_DIR / "told_qp_seed1.pt"),
    "specialist QP 38+": (False, 0, SPECIALIST_QP, RUN_DIR / "specialist_qp38_seed0.pt"),
}

## 1. Train

`training.fit`, the shared recipe. The told model at seeds 0 and 1; the specialist at seed 0, trained and selected on
slice QP 38 and up only. Each run is saved to `checkpoints/qp_runs/` after every epoch.

In [2]:
runs = {}
for name, (told, seed, qp_range, path) in RUNS.items():
    model = build(told, seed)
    history, best = training.fit(model, path, manifest, told=TOLD if told else None, seed=seed, qp_range=qp_range,
                                 verbose=0)
    runs[name] = model.eval()
    print(f"{name:<22} best epoch {best['epoch']:3} of {len(history):3}, select SSIM-Y {best['score']:.4f}")

train  140795 patches
select   6982 patches, to choose the best epoch
told the QP            best epoch 131 of 143, select SSIM-Y 0.8920
train  140795 patches
select   6982 patches, to choose the best epoch
told the QP, seed 1    best epoch 115 of 127, select SSIM-Y 0.8915
train   33480 patches at slice QP 38-99
select   1585 patches at slice QP 38-99, to choose the best epoch
specialist QP 38+      best epoch 113 of 125, select SSIM-Y 0.7999


## 2. Gain over bicubic

Every model on the whole validation split, then by slice QP bin. The specialist is scored on every bin, but it was
only trained for QP 38 and up.

In [3]:
scores = {}
for name, model in runs.items():
    if RUNS[name][0]:
        scores[name] = training.val_scores(lambda lr, condition: model(lr, condition[:, COLUMNS]), manifest)
    else:
        scores[name] = training.val_scores(lambda lr, condition: model(lr), manifest)
    training.record(NOTEBOOK, name, scores[name])

blind_mean = {key: np.mean([s[key] for s in blind.values()], axis=0) for key in ("ssim", "y")}
training.show(training.compare({"bicubic": bicubic, **blind, **scores}, manifest, base=bicubic))
print()
training.show(training.gain_by_qp({"blind, mean of 3 seeds": blind_mean, **scores}, manifest, base=bicubic))

                     model SSIM-Y over bicubic          95% range luma dB dB over chroma dB over   video  screen   photo    x264   nvenc
               told the QP 0.8914      +0.0393 +0.0371 to +0.0417   30.21   +1.63          +0.05 +0.0230 +0.0578 +0.0249 +0.0417 +0.0370
       told the QP, seed 1 0.8909      +0.0388 +0.0365 to +0.0412   30.17   +1.59          +0.04 +0.0226 +0.0573 +0.0242 +0.0410 +0.0367
        luma 24 + 2 blocks 0.8886      +0.0365 +0.0342 to +0.0389   30.00   +1.42          +0.04 +0.0200 +0.0555 +0.0206 +0.0381 +0.0349
luma 24 + 2 blocks, seed 1 0.8885      +0.0364 +0.0341 to +0.0388   29.99   +1.41          +0.04 +0.0202 +0.0552 +0.0209 +0.0382 +0.0347
luma 24 + 2 blocks, seed 2 0.8883      +0.0362 +0.0340 to +0.0386   29.98   +1.40          +0.04 +0.0199 +0.0551 +0.0207 +0.0379 +0.0346
         specialist QP 38+ 0.8765      +0.0244 +0.0220 to +0.0267   28.64   +0.05          +0.03 +0.0123 +0.0410 +0.0049 +0.0293 +0.0196
                   bicubic 0.8521      +0

### More than luck?

By slice QP bin, in SSIM-Y: the told model's lead over the blind one (the mean of its two seeds against the mean of
the blind model's three), how far apart the blind seeds and the two told seeds land in that bin, and the specialist
against the told model.

In [4]:
qp = np.load(DATA_DIR / manifest["splits"]["val"]["qp"])
edges = manifest["qp_edges"]
told_names = ["told the QP", "told the QP, seed 1"]
print(f"{'QP':>7}{'patches':>9}{'told - blind':>14}{'blind seeds':>13}{'told seeds':>12}{'specialist - told':>19}")
for low, high in zip(edges[:-1], edges[1:]):
    inside = (qp >= low) & (qp < high)
    mean_in = lambda s: s["ssim"][inside].mean()
    told_mean = np.mean([mean_in(scores[name]) for name in told_names])
    blind_in = [mean_in(s) for s in blind.values()]
    print(f"{f'{low}-{high}':>7}{inside.sum():9}{told_mean - np.mean(blind_in):+14.4f}{np.ptp(blind_in):13.4f}"
          f"{abs(mean_in(scores[told_names[0]]) - mean_in(scores[told_names[1]])):12.4f}"
          f"{mean_in(scores['specialist QP 38+']) - told_mean:+19.4f}")

     QP  patches  told - blind  blind seeds  told seeds  specialist - told
   0-18     3610       +0.0050       0.0001      0.0003            -0.0392
  18-26     3928       +0.0020       0.0003      0.0003            -0.0286
  26-30     3730       +0.0007       0.0005      0.0004            -0.0199
  30-34     3585       +0.0007       0.0004      0.0005            -0.0100
  34-38     3539       +0.0019       0.0002      0.0006            -0.0024
  38-42     2846       +0.0042       0.0002      0.0007            +0.0022
  42-46     2005       +0.0053       0.0002      0.0007            +0.0033
  46-52     1043       +0.0057       0.0001      0.0010            +0.0024


## 3. What it says

- **The slice QP is worth telling.** Told it, notebook 7's body scores 0.8914 and 0.8909 at two seeds (+0.0393 and
  +0.0388 over bicubic, 0.0005 apart): +0.0027 over the blind model's three seeds (0.8883 to 0.8886). Notebook 8's
  model, told the bitrate and the line instead, gained +0.0020, and at the same seed the QP is +0.0010 above it.
  Shipped as the blind graph with its biases a line in the QP (notebook 19), it costs the client nothing at run time.
- **It pays most at both ends of the QP range**: +0.0050 at QP 0–18 and +0.0053 to +0.0057 from 42 up, against
  +0.0007 from 26 to 34. In every bin the lead is wider than both seed spreads there (the blind seeds land at most
  0.0005 apart in a bin, the told ones 0.0010).
- **The specialist wins where it was trained and loses everywhere else.** Trained and selected only on slice QP 38
  and up (33,480 patches, one seed), it beats the told model there by +0.0022 to +0.0033 a bin, two to three times the
  told seeds' spread. Below QP 34 it is 0.010 to 0.039 behind, and at QP 0–18 below bicubic (−0.0013).
- **A second graph for high QP is worth little.** Switched in at QP 38, the cut it was trained for, the specialist
  would lift the told model's 0.8914 to 0.8920 overall: +0.0005, for a second graph the client has to load and
  switch between. Notebook 20 asks the same of two told models.